In [2]:
from ZV_functions import *
import tensorflow as tf
from matplotlib import pyplot as plt
import matplotlib
import pandas as pd
from keras.layers import Dense, Dropout, LeakyReLU
from keras.models import Sequential
from keras.optimizers import Adam, SGD
from keras.callbacks import EarlyStopping
from keras.constraints import maxnorm
import keras.backend as k
from tensorflow.keras import regularizers
matplotlib.rcParams.update({'font.size': 16})
from sklearn.model_selection import train_test_split

Using TensorFlow backend.


# Bayesian optimization of DNN architecture

In [2]:
#Python files location (structure= base_dir+'/Numpy/ZV_'+year+'/'+cut+'/sample/v1/*.pkl')
base_dir='D:\\Travail\\ZV_analysis\\'
years=['2018_btag', '2017_btag', '2016_btag']
cut='Resolved_SR_bVeto'
vers=''
#outputdir
output_dir='All_years_noqgl'
#training version
training = 'Pruned_noqgl' #'Pruned'

#list of features depend on training version and cut
if training == 'Pruned':
    if 'Boosted' in cut:
        features =['mjj_max', 'vbs_jet_qgl1', 'vbs_jet_qgl2', 
                   'FatJet_pt', 'vbs_jet_eta2', 
                   'FatJeteta', 'vbs_jet_eta1',  'vbs_jet_pt2',  'Zlep_1', 'detajj'

                  ] # , 'Zlep_2', 'mll',, 'Lep_pt1', 'vbs_jet_pt1','Lep_pt2',
        #' 'Zvjet', 'dphijj_mjjmax',njet', 'vbs_jet_pt2', 'Lep_eta2', 

    elif 'Resolved' in cut:
        features=['mjj_max', 'V_jet_qgl1','V_jet_qgl2',  
                 'vbs_jet_qgl1', 'Zlep_1', 'vbs_jet_qgl2',
                 'Zlep_2', 'V_jet_eta1', 'V_jet_eta2','dphijj', 
                  'vbs_jet_pt2','vbs_jet_pt1','V_jet_pt2', 'detajj'

                 ]# 'Lep_eta2','V_jet_mass','V_jet_pt1','Lep_eta1','vbs_jet_pt1','V_jet_pt2',
    #'dphijj_mjjmax', 'vbs_jet_pt2', 'detajj_mjjmax',

elif training == "Full":
    if 'Boosted' in cut:
        features =['Lep_pt1', 'Lep_pt2', 'Lep_eta1',
           'Lep_eta2', 'mll',
           'FatJet_pt', 'FatJeteta',
           'Zlep_1', 'Zlep_2', 
           'vbs_jet_pt1', 'vbs_jet_pt2',
           'vbs_jet_eta1', 'vbs_jet_eta2',
           'mjj_max', 'detajj', 'dphijj', 
            'njet', 'Zvjet', 'vbs_jet_qgl1', 'vbs_jet_qgl2']
    elif 'Resolved' in cut:
        features=['Lep_pt1', 'Lep_pt2', 'Lep_eta1',
           'Lep_eta2', 'mll',
           'Zlep_1', 'Zlep_2', 
           'vbs_jet_pt1', 'vbs_jet_pt2',
           'vbs_jet_eta1', 'vbs_jet_eta2',
           'V_jet_pt1', 'V_jet_pt2',
           'V_jet_eta1', 'V_jet_eta2',
           'mjj_max', 'detajj', 'dphijj',
           'V_jet_mass', 'njet', 'vbs_jet_qgl1', 'vbs_jet_qgl2',
             'V_jet_qgl1', 'V_jet_qgl2']
elif training == "Full_btag":
    if 'Boosted' in cut:
        features =['Lep_pt1', 'Lep_pt2', 'Lep_eta1',
           'Lep_eta2', 'mll',
           'FatJet_pt', 'FatJeteta',
           'Zlep_1', 'Zlep_2', 
           'vbs_jet_pt1', 'vbs_jet_pt2',
           'vbs_jet_eta1', 'vbs_jet_eta2',
           'mjj_max', 'detajj', 'dphijj', 
            'njet', 'Zvjet', 'vbs_jet_qgl1', 'vbs_jet_qgl2', 'nbtag']
    elif 'Resolved' in cut:
        features=['Lep_pt1', 'Lep_pt2', 'Lep_eta1',
           'Lep_eta2', 'mll',
           'Zlep_1', 'Zlep_2', 
           'vbs_jet_pt1', 'vbs_jet_pt2',
           'vbs_jet_eta1', 'vbs_jet_eta2',
           'V_jet_pt1', 'V_jet_pt2',
           'V_jet_eta1', 'V_jet_eta2',
           'mjj_max', 'detajj', 'dphijj',
           'V_jet_mass', 'njet', 'vbs_jet_qgl1', 'vbs_jet_qgl2',
             'V_jet_qgl1', 'V_jet_qgl2', 'nbtag']
        
elif training == "Full_noqgl":
    if 'Boosted' in cut:
        features =['Lep_pt1', 'Lep_pt2', 'Lep_eta1',
           'Lep_eta2', 'mll',
           'FatJet_pt', 'FatJeteta',
           'Zlep_1', 'Zlep_2', 
           'vbs_jet_pt1', 'vbs_jet_pt2',
           'vbs_jet_eta1', 'vbs_jet_eta2',
           'mjj_max', 'detajj', 'dphijj', 
            'njet', 'Zvjet',]
    elif 'Resolved' in cut:
        features=['Lep_pt1', 'Lep_pt2', 'Lep_eta1',
           'Lep_eta2', 'mll',
           'Zlep_1', 'Zlep_2', 
           'vbs_jet_pt1', 'vbs_jet_pt2',
           'vbs_jet_eta1', 'vbs_jet_eta2',
           'V_jet_pt1', 'V_jet_pt2',
           'V_jet_eta1', 'V_jet_eta2',
           'mjj_max', 'detajj', 'dphijj',
           'V_jet_mass', 'njet', ]

elif training == "Pruned_noqgl":
    if 'Boosted' in cut:
        features =['Lep_eta1',
           'Lep_eta2',
           'FatJet_pt', 
           'Zlep_1',
           'vbs_jet_pt2',
            'vbs_jet_eta2',
           'mjj_max', 'detajj',
            'njet', 'Zvjet']
    elif 'Resolved' in cut:
        features=[        'Zlep_1',
           'V_jet_eta1', 'V_jet_eta2',
           'mjj_max', 'detajj', 'njet', ]
        
elif training == "Pruning":
    if 'Boosted' in cut:
        features =[    
           
           'mjj_max', 'detajj', 
             ] #'Lep_pt2','mll','Lep_pt1','vbs_jet_pt1','FatJeteta','vbs_jet_eta1','dphijj', 'Zlep_2', 
        # 'FatJet_pt', 'vbs_jet_eta2','Lep_eta1',  'vbs_jet_pt2','Zvjet''Zlep_1',  'Lep_eta2','njet',
    elif 'Resolved' in cut:
        features=[
            
           'mjj_max', 'detajj', 
            ] #'mll', 'Lep_pt2', 'Lep_eta2','vbs_jet_pt1','Lep_pt1',  'vbs_jet_eta1', 'vbs_jet_eta2','Lep_eta1','V_jet_mass','V_jet_pt1',,
                        #'dphijj','V_jet_pt2',            'vbs_jet_pt2','Zlep_2', 'njet', 'V_jet_eta2','Zlep_1', 'V_jet_eta1',

#save preprocessed data files to data_dir    
data_dir=base_dir+name+'\\'+cut+'\\data'+vers
input_list= features

#add 'cut' to the features name to replicate latinos output
inputs= [cut+'_'+i for i in input_list]



# Preprocesssing 

In [3]:
#preprocess samples for training
X,y, variables_list=prep_data_multi(years,name, cut, vers, base_dir) #

plot_feats=False

if plot_feats:
    #x (min,max) for plotting
    xlims=[(0,200),(0,150),(-3,3),(-3,3),(0,150),(0,5),(0,10),(0,900),(
        -5,5),(0,200),(0,1),(-1.5,1.5),(-1.5,1.5),(-1,1),
           (0,200),(0,200),(-5,5),(-5,5),
               (0,200),(0,100),(-5,5),(-5,5),(0,3000),(0,10),(0,4),(0,150)]

    plot_distrib(variables_list, xlims,base_dir, year, cut)
    
    
#Number of event for signal/bkg
print('Number of events: {}, signal : {}, bkg: {}'.format(len(y),len(y[y['signal']==1]),len(y[y['signal']==0])))
#Yields (sum of weights) for signal/bkg
print('SOW: {}, signal : {}, bkg: {}'.format(sum(y['weight_']),sum(y['weight_'][y['signal']==1]),sum(y['weight_'][y['signal']==0])))

X= pd.DataFrame(np.load(data_dir+'\\X_{}_{}{}.npy'.format(name,cut,vers), allow_pickle=True), columns=variables_list)
X=X[inputs]
y= pd.DataFrame(np.load(data_dir+'\\y_{}_{}{}.npy'.format(name,cut,vers), allow_pickle=True), columns= ['year', 'signal', 'sample','group','weight_', 'w'])

#80% train/20% test splitting
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2, random_state=42)



2018_btag
2017_btag
2016_btag
Resolved_SR_bVeto_Lep_pt1 118.33808 87.34349
Resolved_SR_bVeto_Lep_pt2 51.668495 34.46581
Resolved_SR_bVeto_Lep_eta1 -0.0021136126 1.2356488
Resolved_SR_bVeto_Lep_eta2 -0.0031651622 1.2348542
Resolved_SR_bVeto_mll 90.771866 4.146833
Resolved_SR_bVeto_Zepp_ll 0.5418927 0.4050713
Resolved_SR_bVeto_nFatJet 0.0 0.0
Resolved_SR_bVeto_nCleanFatJet 0.0 0.0
Resolved_SR_bVeto_FatJet_pt -999.0 0.0
Resolved_SR_bVeto_FatJeteta -999.0 0.0
Resolved_SR_bVeto_FatJet_softdropmass 0.0 0.0
Resolved_SR_bVeto_FatJet_tau21 -999.0 0.0
Resolved_SR_bVeto_Zlep_1 -0.0008862076 0.38887227
Resolved_SR_bVeto_Zlep_2 -0.0010090704 0.38741916
Resolved_SR_bVeto_category 1.0 0.0
Resolved_SR_bVeto_vbs_jet_pt1 186.32042 128.0351
Resolved_SR_bVeto_vbs_jet_pt2 78.53296 59.03812
Resolved_SR_bVeto_vbs_jet_eta1 0.0015283659 2.1661296
Resolved_SR_bVeto_vbs_jet_eta2 0.0023261367 2.8171685
Resolved_SR_bVeto_V_jet_pt1 86.93671 59.487503
Resolved_SR_bVeto_V_jet_pt2 43.31726 14.421122
Resolved_SR_bVeto_

In [5]:
# DNN training function
def build_dnn(base_dir, 
              lr, #learning rate
              l1, # L1 regu
              l2, # L2 rregu
              dropout, #dropout function (same for each hidden layer)
              sl1, #neurons in HL 1
              sl2, #neurons in HL 2
              sl3, #neurons in HL 3
              sl4, #neurons in HL 4
              sl5#neurons in HL 5
             ):
    from tensorflow.keras.layers import BatchNormalization

    SoB_thr=0.8
   
    
    NN_name='DNN_{}_{}_{}_{}_{}_lr{}_dp{}'.format(sl1,sl2,sl3,sl4,sl5,lr,dropout)
    reg=regularizers.L1L2(l1,l2)
    opt=Adam(lr=lr)
    #opt=SGD(lr=lr, momentum=0.99)


    DNN=Sequential()
    act_func=tfunc
    #,kernel_constraint=maxnorm(3)
    DNN.add(Dense(sl1,input_dim=len(inputs),activation=act_func,kernel_regularizer=reg))
    DNN.add(BatchNormalization())
    DNN.add(Dropout(rate=dropout))
    

    if sl2!=0:
        DNN.add(Dense(sl2,activation=act_func,kernel_regularizer=reg))
        DNN.add(BatchNormalization())
        DNN.add(Dropout(rate=dropout))
        
    if sl3!=0:
        DNN.add(Dense(sl3,activation=act_func,kernel_regularizer=reg))
        DNN.add(BatchNormalization())
        DNN.add(Dropout(rate=dropout))
        
    if sl4!=0:
        DNN.add(Dense(sl4,activation=act_func))
        DNN.add(BatchNormalization())
        DNN.add(Dropout(rate=dropout))
        
    if sl5!=0:
        DNN.add(Dense(sl5,activation=act_func))
        DNN.add(BatchNormalization())
        DNN.add(Dropout(rate=dropout))
        


    DNN.add(Dense(1,activation='sigmoid'))
    auc=tf.keras.metrics.AUC()
    DNN.compile(loss=loss,optimizer=opt ) 
    
    return DNN, NN_name

# Bayesian optimization

In [7]:
import bayes_opt
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, LeakyReLU
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.constraints import MaxNorm as maxnorm
import keras.backend as k
from tensorflow.keras import regularizers
from sklearn.metrics import roc_curve, auc

#scoring function: test auc penalized by divergence train/test
def Score(roc_auc, roc_auc_train):
    alpha=1
    return roc_auc-alpha*(np.abs(roc_auc-roc_auc_train))
    #function to optimize in optimization
    
# function to optimize: train DNN with given parameters and returns its score 
#can add other optimizable parameters here 
def Score_dnn(n_layer, nl1,nl2, nl3, nl4, nl5, lr, l2, dropout):
    l1=0
    #l2=0.001
    #dropout=0.2
    n_layer=int(n_layer)
    sl1=int(nl1)
    sl2,sl3,sl4,sl5 = 0,0,0,0
    if n_layer >= 2:
        sl2=int(nl2)
    if n_layer >= 3:
        sl3=int(nl3)
    if n_layer >= 4:
        sl4=int(nl4)
    if n_layer >= 5:
        sl5=int(nl5)
    
    base_dir='D:\\Travail\\ZV_analysis\\'
    print('training network with [{}_{}_{}_{}_{}] neurons'.format(sl1,sl2,sl3,sl4,sl5))
    DNN, res_dir=build_dnn(base_dir, lr, l1, l2, dropout, sl1, sl2, sl3, sl4, sl5)
    
    es_callback = EarlyStopping(monitor='val_loss', patience=early_stop, restore_best_weights=True)
    history=DNN.fit(x=X_train, y=y_train['signal'].to_numpy(),
                    sample_weight=y_train['w'].to_numpy(), epochs=epochs, batch_size=batchsize, 
                    verbose=0, callbacks=[es_callback], 
                    validation_data=(X_test, y_test['signal'].to_numpy(),y_test['w'].to_numpy()))

    predictions_NN= DNN.predict(X_test)
    predictions_NN_train=DNN.predict(X_train)
    fpr, tpr, threshold = roc_curve(y_test['signal'].astype('int32'),predictions_NN, pos_label=1, sample_weight=y_test['w'])
    tpr.sort()
    fpr.sort()
    roc_auc =auc(fpr, tpr)
    fpr, tpr, threshold = roc_curve(y_train['signal'].astype('int32'),predictions_NN_train, pos_label=1, sample_weight=y_train['w'])
    tpr.sort()
    fpr.sort()
    roc_auc_train =auc(fpr, tpr)
    return Score(roc_auc, roc_auc_train)

In [8]:
#optimize lr, l1,l2, dropout, sls

#train on gpu
physical_devices = tf.config.list_physical_devices('GPU')
print(physical_devices)

predictions_NN={}
target={}


early_stop=3  #number of epochs without gain before stoping
epochs=200 #max epochs
batchsize=1024
tfunc='relu'
loss='binary_crossentropy'


#hyperparameters bounds
pbounds={'n_layer': (1, 5),
         'nl1': (20, 300), 
         'nl2': (0, 300),
         'nl3': (0, 300),
         'nl4': (0, 300),
         'nl5': (0, 300),
         'lr': (0.0001, 0.001),
         'l2' : (0.0001, 0.1),
         'dropout' : (0,0.5),
        }


optimizer = bayes_opt.BayesianOptimization(
    f=Score_dnn,
    pbounds=pbounds,
    verbose=2, # verbose = 1 prints only when a maximum is observed, verbose = 0 is silent
    random_state=1,
)

In [9]:
optimizer.maximize(init_points=40, n_iter=60)
optimizer.max

|   iter    |  target   |  dropout  |    l2     |    lr     |  n_layer  | size_l... |
-------------------------------------------------------------------------------------
training network with [78_78_0_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  1        |  0.8322   |  0.2085   |  0.07206  |  0.000100 |  2.209    |  78.16    |
training network with [180_180_0_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  2        |  0.8267   |  0.04617  |  0.01871  |  0.000411 |  2.587    |  180.1    |
training network with [47_47_47_47_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  3        |  0.8287   |  0.2096   |  0.06855  |  0.000284 |  4.512    |  47.12    |
training network with [91_0_0_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  4        |  0.8273   |  0.3352   |  0.04179  |  0.000602 |  1.562    |  91.51    |
training network with [267_267_267_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  5   

  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  24       |  0.8137   |  0.3777   |  0.07541  |  0.000930 |  3.846    |  72.31    |
training network with [263_0_0_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  25       |  0.8275   |  0.00994  |  0.002718 |  0.000125 |  1.985    |  263.6    |
training network with [112_0_0_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  26       |  0.8257   |  0.2694   |  0.05533  |  0.000857 |  1.497    |  112.6    |
training network with [248_0_0_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  27       |  0.8249   |  0.2929   |  0.09696  |  0.000604 |  1.075    |  248.2    |
training network with [234_234_234_234_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  28       |  0.8249   |  0.1165   |  0.08073  |  0.000449 |  4.454    |  234.3    |
training network with [51_0_0_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  29       |  0.8304   |  0.2781   |  0.0137

|  47       |  0.8279   |  0.2752   |  0.02148  |  0.000984 |  2.705    |  254.5    |
training network with [145_145_145_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  48       |  0.8255   |  0.3307   |  0.05047  |  0.000335 |  3.489    |  145.8    |
training network with [224_224_0_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  49       |  0.8263   |  0.07042  |  0.02676  |  0.000865 |  2.796    |  224.7    |
training network with [161_161_161_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  50       |  0.8316   |  0.2345   |  0.003862 |  0.000217 |  3.823    |  161.8    |
training network with [56_0_0_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  51       |  0.8311   |  0.1923   |  0.05328  |  0.000127 |  1.535    |  56.06    |
training network with [121_121_121_121_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  52       |  0.8268   |  0.1341   |  0.05409  |  0.000508 |  4.911    |  121.6    |

  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  71       |  0.8307   |  0.4917   |  0.00889  |  0.00015  |  2.446    |  289.0    |
training network with [220_220_220_220_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  72       |  0.8272   |  0.05891  |  0.08602  |  0.000310 |  4.578    |  220.6    |
training network with [195_195_195_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  73       |  0.8276   |  0.4826   |  0.002974 |  0.000616 |  3.314    |  195.4    |
training network with [206_0_0_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  74       |  0.8312   |  0.419    |  0.07206  |  0.000110 |  1.504    |  206.5    |
training network with [65_0_0_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  75       |  0.8255   |  0.01782  |  0.0944   |  0.000763 |  1.651    |  65.16    |
training network with [137_137_137_137_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  76       |  0.8222   |  0.2583  

  ...
    to  
  ['...']
|  94       |  0.8247   |  0.1821   |  0.02843  |  0.000822 |  3.506    |  141.2    |
training network with [291_291_291_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  95       |  0.8113   |  0.2696   |  0.09154  |  0.000491 |  3.469    |  292.0    |
training network with [58_58_58_58_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  96       |  0.8248   |  0.118    |  0.0814   |  0.000617 |  4.157    |  58.25    |
training network with [176_176_176_176_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  97       |  0.8294   |  0.05619  |  0.02389  |  0.000133 |  4.445    |  176.9    |
training network with [127_127_127_127_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  98       |  0.8105   |  0.46     |  0.06318  |  0.000838 |  4.696    |  127.0    |
training network with [180_180_180_0_0] neurons
  ...
    to  
  ['...']
  ...
    to  
  ['...']
|  99       |  0.8291   |  0.06368  |  0.008902 |  0.

{'target': 0.8329421871931695,
 'params': {'dropout': 0.44730333175192366,
  'l2': 0.008595916715840812,
  'lr': 0.00013514930490959414,
  'n_layer': 1.6793216782582756,
  'size_layer': 268.31705089164745}}

In [ ]:
res=optimizer.res
df=pd.DataFrame()
df['target']=[item['target'] for item in res]
df['l2']=[item['params']['l2'] for item in res]
df['lr']=[item['params']['lr'] for item in res]
df['dp']=[item['params']['dropout'] for item in res]
df['n_layer']=[item['params']['n_layer'] for item in res]
df['size_layer']=[item['params']['size_layer'] for item in res]

best=df.sort_values(['target'],ascending=False).head(10)

best